In [1]:
import cv2, numpy as np, matplotlib.pyplot as plt, glob
from tqdm import tqdm
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

initial_targets = sorted(glob.glob("./subset/*"))
images = glob.glob("/Volumes/SD_Card/DCIM/DJI_20260401*/*.JPG")

target = initial_targets[-5]
# Image.open(target)

In [2]:
def get_exif_data(image_path):
    image = Image.open(image_path)
    exif_data = image._getexif()
    
    if not exif_data:
        return None
    
    exif = {}
    for tag, value in exif_data.items():
        decoded = TAGS.get(tag, tag)
        exif[decoded] = value
    
    return exif

import subprocess
import json

def get_gimbal_pitch(image_path):
    cmd = [
        "exiftool",
        "-json",
        "-GimbalPitchDegree",
        "-CameraPitch",
        image_path
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    data = json.loads(result.stdout)[0]
    
    return data.get("GimbalPitchDegree") or data.get("CameraPitch")

def get_coords(path):
    exif = get_exif_data(path)
    gps = exif['GPSInfo']
    lat = gps[2]

    lon = gps[4]
    if gps[3] == 'W':
        lon = [-l for l in lat]
    return np.array([
        lat,
        lon
    ]).astype(np.float64)
    
target_coords = get_coords(target)
target_coords

array([[ 25.    ,  43.    ,  28.6236],
       [-25.    , -43.    , -28.6236]])

In [3]:
im = np.asarray(Image.open(target))
T = np.mean(im, axis=-1)
mask = np.linalg.norm(im / 255 - np.array([1, 1, 1]), axis=-1) < 0.5
mask = cv2.medianBlur(mask.astype(np.uint8), 15)

KeyboardInterrupt: 

In [ ]:
plt.imshow(mask)
plt.axis('off')

In [ ]:
mask2 = cv2.pyrUp(cv2.pyrUp(
    cv2.dilate(
        cv2.medianBlur(cv2.pyrDown(cv2.pyrDown(mask)), 15),
        np.ones((11, 11))
    )
))[:mask.shape[0], :mask.shape[1]]
mask2 = 1 - (cv2.dilate(mask2, np.ones((17, 17))))
plt.imshow((mask2 * 0.75 + 0.25)[..., np.newaxis] * im / 255)

In [ ]:
# Edges
mask3 = np.invert(cv2.GaussianBlur(cv2.Canny(im, 100, 200), [3, 3], 3) > 0)
plt.imshow(mask3)

In [ ]:
final_mask = np.bitwise_and(mask, np.bitwise_and(mask2, mask3))
plt.imshow(final_mask)

# cv2.imshow("Circles", im * ((cv2.GaussianBlur(final_mask, [15, 15], 15) > 0)[..., np.newaxis] * 0.5 + 0.5) / 255)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
lines = cv2.HoughLinesP(
    final_mask,
    rho=1,
    theta=np.pi / 180,
    threshold=10,
    minLineLength=50,
    maxLineGap=10
)

print(lines.shape)
img = im.copy() // 2
# Draw lines

line_mask = np.ones_like(final_mask)
X, Y = np.meshgrid(np.arange(final_mask.shape[1]), np.arange(final_mask.shape[0]))
print(Y.max(), X.max())
# print("???")
dist_thresh = 10
coords = np.dstack([Y, X])
if lines is not None:
    for line in tqdm(lines):
        x1, y1, x2, y2 = line[0]
        p1 = np.array([y1, x1]).astype(np.float32)
        p2 = np.array([y2, x2]).astype(np.float32)
        v = p2 - p1
        L = np.linalg.norm(v)
        v /= L
        V = coords - p1
        t = np.vecdot(V, v)
        P = np.linalg.norm(V - v * t[..., np.newaxis], axis=-1)
        M = np.invert(np.bitwise_and(P < dist_thresh, np.bitwise_and(t > dist_thresh, t < L + dist_thresh)))
        line_mask = np.bitwise_and(line_mask, M)
        
        cv2.line(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
# plt.imshow(img)
# plt.imshow(M)

In [ ]:
# lines[0]
# print("Hi")
plt.imshow((line_mask * 0.75 + 0.25)[..., np.newaxis] * im / 255)

In [ ]:
final_mask = np.bitwise_and(final_mask, line_mask)
plt.imshow(final_mask)

In [ ]:
# mask = cv2.GaussianBlur(mask, [15, 15], 15)

In [ ]:
circles = cv2.HoughCircles(
    final_mask.astype(np.uint8),
    cv2.HOUGH_GRADIENT,
    dp=2,
    minDist=20,
    param1=5,   # edge detection threshold
    param2=20,    # circle detection sensitivity
    minRadius=5,
    maxRadius=30
)

In [ ]:
circles

In [ ]:
img = np.asarray(Image.open(target)).astype(np.uint8)
if circles is not None:
    C = np.round(circles[0, :]).astype("int")
    for (x, y, r) in C:
        cv2.circle(img, (x, y), r, (0, 255, 0), 2)
        cv2.circle(img, (x, y), 2, (0, 0, 255), 3)
plt.imshow(img)
plt.axis('off')
plt.tight_layout()

cv2.imshow("Circles", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
img = np.repeat(mask.copy(), 3).reshape((im.shape)).astype(np.uint8) * 255
if circles is not None:
    C = np.round(circles[0, :]).astype("int")
    for (x, y, r) in C:
        cv2.circle(img, (x, y), r, (0, 255, 0), 2)
        cv2.circle(img, (x, y), 2, (0, 0, 255), 3)
plt.imshow(img)
plt.axis('off')
plt.tight_layout()

# cv2.imshow("Circles", img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
im = np.asarray(Image.open(target))
# mask = np.linalg.norm(im / 255 - np.array([1, 1, 1]), axis=-1) < 0.5
gray = im / np.maximum(1e-4, np.linalg.norm(im, axis=-1))[..., np.newaxis]
plt.imshow(np.linalg.norm(gray - np.array([1, 1, 1]) * np.sqrt(3), axis=-1)[:, 2000:])

In [ ]:
L = 4
L = np.arange(L * 4) - L * 2
X, Y = np.meshgrid(L, L)
kernel = ((X * X + Y * Y) < (8 * 8)) * 2 - 1
kernel = kernel / np.sum(kernel)

p = cv2.filter2D(mask.astype(np.float32), cv2.CV_32F, kernel)

In [ ]:
kernel.shape

In [ ]:
plt.imshow(T[2000:2100, 2000:2100])

In [ ]:
np.max(T[2000:2100, 2000:2100] / 255), np.min(T[2000:2100, 2000:2100] / 255)

In [ ]:
plt.imshow(np.hstack([T[2000:2100, 2000:2100] / 255, p[2000:2100, 2000:2100]]))
plt.colorbar()

In [ ]:
plt.imshow(p)

In [ ]:
def find_pins(image):
    

In [ ]:
# targets = []
# dist = []
# for t in tqdm(initial_targets):
#     d = np.sum(np.abs(get_coords(t) - target_coords) * [60 * 60, 60, 1])
#     print(d, get_coords(t) - target_coords)
#     if d < 0.4:
#         gimb = get_gimbal_pitch(t)
#         if np.abs(gimb + 90) < 0.2:
#             targets.append(t)
#             dist.append(d)
#         else:
#             print("Rejecting target image due to gimball", gimb, t)
# print(f"Found {len(targets)} targets, which is {len(targets) / len(images) * 100:.2f}% of the data")

In [ ]:
# np.argmax(dist), np.max(dist)

In [ ]:
# T = np.asarray(Image.open(target))
# O = np.asarray(Image.open(targets[np.argmax(dist)]))

In [ ]:
# plt.imshow(
#     np.dstack([
#         np.mean(T, axis=-1) / 255,
#         np.mean(O, axis=-1) / 255,
#         np.mean(T, axis=-1) * 0
#     ])
# )

In [ ]:
# plt.imshow(O)

In [ ]:
# get_exif_data(targets[np.argmax(targets)])

In [ ]:
# dist